<a href="https://colab.research.google.com/github/mythien107/busad878/blob/main/Naive_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install dependencies

In [ ]:
!pip install -q anthropic sentence-transformers numpy

# Load the API key

In [ ]:
import os
from google.colab import userdata

# In Colab: click the key icon in the left sidebar and add a secret
# named ANTHROPIC_API_KEY. Then this loads it.
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

import anthropic
client = anthropic.Anthropic()

# Sanity check
print("API key loaded:", os.environ["ANTHROPIC_API_KEY"][:10] + "...")

# Sample document

In [ ]:
# Aurora Manufacturing's fictional PTO policy. We'll use this as our
# "knowledge base" — in production this might be 10,000 SharePoint pages.

POLICY_DOC = """
Aurora Manufacturing — Paid Time Off Policy (Effective January 2026)

1. Eligibility. All full-time U.S. employees are eligible for PTO from
their date of hire. Part-time employees who work at least 24 hours per
week accrue PTO at a prorated rate.

2. Accrual. Employees with less than 3 years of service accrue 15 days
of PTO per year. Employees with 3 to 7 years accrue 20 days. Employees
with more than 7 years of service accrue 25 days. Accrual is monthly,
and unused PTO may be carried over up to a maximum of 10 days.

3. Approval. PTO requests must be submitted at least 14 days in advance
through Workday. Requests for more than 5 consecutive days require
manager and HR approval. Requests during fiscal year-end close
(December 15 to January 5) are restricted to emergencies only.

4. Sick leave. Sick leave is separate from PTO. Employees receive 10
sick days per year, non-cumulative. Sick days do not require advance
notice but must be reported to the manager by 9:00 AM on the day of
absence.

5. Bereavement. Up to 5 days of paid bereavement leave is provided for
the loss of an immediate family member; up to 2 days for an extended
family member.

6. Parental leave. Aurora provides 16 weeks of paid parental leave for
the birth or adoption of a child, available to either parent.

7. Cash-out. Upon separation from the company, accrued and unused PTO
is paid out at the employee's final base salary rate, subject to
applicable taxes and a maximum of 30 days.
"""
print("Document loaded:", len(POLICY_DOC), "characters")

# Chunk the document

In [ ]:
chunks = [c.strip() for c in POLICY_DOC.split("\n\n") if c.strip()]
print(f"Number of chunks: {len(chunks)}")
print("\n--- First chunk ---\n", chunks[1][:200])

# Embed every chunk

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# A small, fast, free embedding model. Production systems use bigger
# models (OpenAI's text-embedding-3-large, Cohere, Voyage, etc.) but
# this is sufficient for a one-page document.
embedder = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = embedder.encode(chunks)
print("Embedding shape:", chunk_embeddings.shape)
# (8, 384) means 8 chunks, each represented by a 384-dimensional vector

# The retrieval function

In [ ]:
def retrieve(query: str, k: int = 3):
    """Return the top-k chunks most similar to the query."""
    query_embedding = embedder.encode([query])[0]

    # Cosine similarity: how aligned are the two vectors?
    similarities = chunk_embeddings @ query_embedding / (
        np.linalg.norm(chunk_embeddings, axis=1) * np.linalg.norm(query_embedding)
    )

    top_indices = np.argsort(similarities)[::-1][:k]
    return [(chunks[i], float(similarities[i])) for i in top_indices]

# Try it
question = "How much PTO do I get if I've been here for five years?"
results = retrieve(question)
for chunk, score in results:
    print(f"[score={score:.3f}]")
    print(chunk[:150])
    print("---")

# Combine retrieval with generation

In [ ]:
def ask(question: str, k: int = 3):
    """Retrieve relevant chunks, then ask Claude to answer using them."""
    retrieved = retrieve(question, k=k)
    context = "\n\n".join([chunk for chunk, _ in retrieved])

    prompt = f"""You are an HR assistant for Aurora Manufacturing.
Answer the user's question using ONLY the policy excerpts below.
If the answer is not in the excerpts, say "I don't have that information
in the policy. Please contact HR."

POLICY EXCERPTS:
{context}

USER QUESTION: {question}
"""

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=400,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.content[0].text

# Run it
answer = ask("How much PTO do I get if I've been here for five years?")
print(answer)

# Failure Mode

In [ ]:
# Ask a question whose answer is NOT in the policy.
print(ask("What's Aurora's policy on remote work from international locations?"))

In [ ]:
# Same question, but we REMOVE the "if not in excerpts, say I don't know" guardrail.
def ask_unsafe(question: str, k: int = 3):
    retrieved = retrieve(question, k=k)
    context = "\n\n".join([chunk for chunk, _ in retrieved])
    prompt = f"""Answer the user's question using the context.

CONTEXT: {context}

QUESTION: {question}
"""
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=400,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.content[0].text

print(ask_unsafe("What's Aurora's policy on remote work from international locations?"))